# Data Science Case Study: Candidate Resume Search Platform

**Millennium Management — Business Development**  
Submitted by: Osheen Kaul · osheen@gmail.com

---

## Overview

This notebook covers:
1. **Resume parsing** — LLM-powered extraction of structured data from PDF and Word files
2. **Data exports** — JSON and CSV outputs of parsed candidate profiles
3. **Streamlit app** — searchable, filterable candidate platform for BD users
4. **Scalability discussion** — how this extends to production volumes

**Live App (CORE):** [https://millennium-ds-case-study.streamlit.app](https://millennium-ds-case-study.streamlit.app) *(link added after deployment)*

---

## Part 1: Setup & Dependencies

In [ ]:
# Standard library — no additional install needed for these
import os
import json
import csv
import re
import subprocess
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime

# HTTP client for Anthropic API calls
import requests

# Data analysis
import pandas as pd

print('Setup complete')

## Part 2: Resume Parsing via LLM API

### Design decisions

**Why LLM for parsing?**  
Resumes have no fixed schema — fonts, layouts, section names, and date formats all vary. Rule-based parsers break on edge cases constantly. A language model reads resumes the way a recruiter does: it understands context, infers missing information (e.g. geography from a phone prefix), and returns structured output.

**Why cache results instead of parsing live?**  
Parsing is a one-time ETL step, not a query operation. Running LLM inference on every search request would be slow, expensive, and unnecessary. Instead, we parse once and store structured JSON — the Streamlit app then queries the JSON at millisecond speed. This mirrors production architectures where document ingestion is a background pipeline separate from the user-facing search layer.

**Model choice:** Claude Haiku — fast, cost-efficient, and accurate for structured extraction tasks. Full parse of 10 resumes costs under $0.05.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# API key loaded from environment variable — never hardcoded in source
API_KEY   = os.environ.get('ANTHROPIC_API_KEY')
MODEL     = 'claude-haiku-4-5'   # fast + economical for extraction tasks
RESUME_DIR = Path('resumes/')     # folder containing PDF and Word resume files
OUT_JSON  = Path('candidates.json')
OUT_CSV   = Path('candidates.csv')

if not API_KEY:
    raise ValueError('Set ANTHROPIC_API_KEY environment variable before running')

In [ ]:
# ── Extraction prompt ─────────────────────────────────────────────────────────
# Structured prompt that instructs the model to return clean JSON.
# The schema is designed around the filters BD users actually care about.

EXTRACTION_PROMPT = """
You are a resume parser for a hedge fund recruiting platform. Extract structured data
from the resume text below and return ONLY valid JSON — no explanation, no markdown.

{
  "name": "Full name",
  "email": "email or null",
  "location": "City, Country",
  "geography": "one of: US, Europe, Asia-Pacific, Other",
  "languages": ["spoken languages"],
  "education": [{"school": "", "degree": "", "field": "", "year_end": 2020}],
  "top_school": true,
  "credentials": ["CFA", "MBA", "PhD"],
  "experience": [{
    "firm": "", "role": "", "start_year": 2018, "end_year": 2024,
    "current": false, "firm_type": "hedge_fund|asset_manager|investment_bank|...",
    "aum_approx": "$40B", "sector": ["Healthcare"], "strategy": ["Long/Short Equity"]
  }],
  "years_experience": 5,
  "seniority": "Junior (0-3 yrs)|Mid-level (3-7 yrs)|Senior (7+ yrs)",
  "market_type": ["buy_side", "sell_side"],
  "primary_sectors": ["top 1-3 sectors"],
  "primary_strategy": "Fundamental Equity|Quantitative/Systematic|Credit|Macro|...",
  "public_markets": true,
  "private_markets": false,
  "technical_skills": ["Python", "R", "SQL"],
  "currently_employed": true,
  "notable_firms": ["Goldman Sachs", "Citadel"],
  "summary": "2-3 sentence professional summary"
}

Rules:
- top_school = true if Ivy League, MIT, Stanford, LSE, Oxford, Cambridge, IIT, INSEAD,
  Wharton, Chicago Booth, Michigan Ross, MIT Sloan, HBS, HKU
- years_experience = 2026 minus first full-time job year
- currently_employed = true if most recent role has no end date
- Use null for truly unknown fields

Resume text:
"""

In [ ]:
# ── Text extractors ───────────────────────────────────────────────────────────

def extract_text_pdf(path: Path) -> str:
    """Extract plain text from a PDF using pdftotext (poppler)."""
    result = subprocess.run(['pdftotext', str(path), '-'], capture_output=True, text=True)
    return result.stdout.strip()


def extract_text_docx(path: Path) -> str:
    """Extract plain text from a Word document by reading the embedded XML."""
    text_parts = []
    with zipfile.ZipFile(path, 'r') as z:
        with z.open('word/document.xml') as f:
            root = ET.parse(f).getroot()
            ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
            for para in root.iter('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p'):
                runs = para.findall('.//w:t', ns)
                line = ''.join(r.text or '' for r in runs)
                if line.strip():
                    text_parts.append(line.strip())
    return '\n'.join(text_parts)


def call_claude(resume_text: str) -> dict:
    """Send resume text to Claude and return parsed structured data as a dict."""
    response = requests.post(
        'https://api.anthropic.com/v1/messages',
        headers={
            'x-api-key': API_KEY,
            'anthropic-version': '2023-06-01',
            'content-type': 'application/json'
        },
        json={
            'model': MODEL,
            'max_tokens': 1500,
            'messages': [{
                'role': 'user',
                'content': EXTRACTION_PROMPT + resume_text[:8000]  # cap tokens
            }]
        },
        timeout=30
    )
    response.raise_for_status()
    raw = response.json()['content'][0]['text'].strip()
    # Strip markdown code fences if model included them
    raw = re.sub(r'^```(?:json)?\n?', '', raw)
    raw = re.sub(r'\n?```$', '', raw)
    return json.loads(raw)

In [ ]:
# ── Main parse loop ───────────────────────────────────────────────────────────

def parse_all_resumes(resume_dir: Path) -> list:
    """Parse all PDF and Word files in resume_dir, return list of candidate dicts."""
    candidates = []
    files = sorted(list(resume_dir.glob('*.pdf')) + list(resume_dir.glob('*.docx')))

    for i, path in enumerate(files, 1):
        print(f'[{i}/{len(files)}] Parsing: {path.name}')
        try:
            text = extract_text_pdf(path) if path.suffix == '.pdf' else extract_text_docx(path)
            if not text.strip():
                print(f'  ⚠ No text extracted, skipping')
                continue
            data = call_claude(text)
            data['source_file'] = path.name
            data['parsed_at'] = datetime.now().isoformat()
            candidates.append(data)
            print(f'  ✓ {data.get("name", "Unknown")}')
        except json.JSONDecodeError as e:
            print(f'  ✗ JSON parse error: {e}')
        except Exception as e:
            print(f'  ✗ Error: {e}')

    return candidates


# Run the parser
# candidates = parse_all_resumes(RESUME_DIR)
# print(f'\nParsed {len(candidates)} candidates')

# NOTE: Results have already been parsed and saved to candidates.json.
# Load from cache to avoid re-running API calls:
with open('candidates.json') as f:
    candidates = json.load(f)
print(f'Loaded {len(candidates)} parsed candidates from cache')

## Part 3: Parsed Data Review

In [ ]:
# Quick overview of what was extracted
summary_rows = []
for c in candidates:
    summary_rows.append({
        'Name':            c.get('name'),
        'Geography':       c.get('geography'),
        'Strategy':        c.get('primary_strategy'),
        'Seniority':       c.get('seniority'),
        'Yrs Exp':         c.get('years_experience'),
        'Credentials':     ', '.join(c.get('credentials', [])),
        'Top School':      '✓' if c.get('top_school') else '',
        'Employed':        '✓' if c.get('currently_employed') else '',
        'Market Side':     ', '.join(c.get('market_type', [])),
        'Primary Sectors': ', '.join(c.get('primary_sectors', [])),
    })

df = pd.DataFrame(summary_rows)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_columns', 12)
df

## Part 4: Data Exports

In [ ]:
# JSON export — already saved during parsing, shown here for completeness
with open(OUT_JSON, 'w') as f:
    json.dump(candidates, f, indent=2)
print(f'JSON saved: {OUT_JSON} ({OUT_JSON.stat().st_size // 1024} KB)')

# CSV export — flattened key fields
flat_fields = [
    'name', 'email', 'location', 'geography', 'years_experience',
    'seniority', 'primary_strategy', 'currently_employed',
    'top_school', 'public_markets', 'private_markets', 'source_file'
]
list_fields = ['languages', 'credentials', 'primary_sectors', 'technical_skills', 'notable_firms', 'market_type']

with open(OUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=flat_fields + list_fields)
    writer.writeheader()
    for c in candidates:
        row = {k: c.get(k, '') for k in flat_fields}
        for lf in list_fields:
            val = c.get(lf, [])
            row[lf] = ', '.join(val) if isinstance(val, list) else str(val)
        writer.writerow(row)

print(f'CSV saved: {OUT_CSV}')
pd.read_csv(OUT_CSV).head()

## Part 5: Streamlit Application

The full application code lives in `app.py` in this repository and is deployed at the link above.

### Filter design rationale

Filters were chosen based on how a Millennium BD recruiter would actually scope a search:

| Filter | Rationale |
|---|---|
| **Free text search** | Search across all fields — find "Goldman" or "machine learning" instantly |
| **Geography** | US / Europe / APAC — maps to regional hiring mandates |
| **Investment strategy** | Fundamental vs. Quant — very different skill profiles |
| **Sector coverage** | Healthcare, Tech, Credit — matches pod-specific JDs |
| **Seniority + years exp.** | Junior vs. senior analyst openings |
| **Buy-side / sell-side** | Different research styles; some pods prefer one background |
| **Credentials** | CFA, MBA, PhD — minimum requirements on many JDs |
| **Technical skills** | Python/R/C++ signals quant capability |
| **Additional languages** | Critical for EM and cross-border mandates |
| **Top school** | Proxy for pedigree in traditional hedge fund recruiting |
| **Currently employed** | Availability signal |
| **Private markets exp.** | Relevant for credit/opportunistic pods |

In [ ]:
# The full Streamlit app code — copy of app.py included here for review

app_code = open('app.py').read()
print(f'app.py: {len(app_code)} characters, {app_code.count(chr(10))} lines')
print('\nTo run locally:')
print('  pip install streamlit plotly pandas requests')
print('  streamlit run app.py')

## Part 6: If More Time — Additional Features & Scalability

### Features I'd add with more time

**1. Semantic / vector search**  
Right now free-text search is substring matching. With more time, I'd embed each candidate profile using a sentence transformer or the Anthropic embeddings API, store vectors in a lightweight store like ChromaDB or pgvector, and enable queries like "candidates similar to our best healthcare analyst" — returning ranked results by semantic similarity rather than exact keyword match. This is meaningfully more powerful for talent search.

**2. Resume scoring against a JD**  
Accept a job description as input, score each candidate on fit (0-100), and surface a ranked shortlist. The LLM would compare JD requirements against candidate profiles and explain its reasoning. This turns the tool from a filter into an active recruiter.

**3. Duplicate / alias detection**  
At scale, the same person may appear multiple times (different resume versions, different formats). Fuzzy name + email matching plus embedding similarity would flag likely duplicates before they pollute the database.

**4. Candidate timeline and status tracking**  
Add pipeline stages (Sourced → Screened → Interview → Offer) so BD can manage candidates through the process inside the same tool rather than switching to a spreadsheet.

**5. Email / LinkedIn integration**  
Auto-populate candidates from inbound email attachments or LinkedIn exports — eliminating the manual upload step.

---

### Scaling to large resume volumes

The current architecture handles tens of resumes well. Here's how I'd scale it:

**Async parsing pipeline**  
Replace the synchronous loop with an async queue (e.g. Celery + Redis or AWS SQS). Each uploaded resume goes into the queue; workers call the LLM API in parallel. This handles thousands of resumes without blocking the UI.

**Persistent database**  
Swap JSON file storage for PostgreSQL with pgvector. This enables SQL filtering (much faster at scale), vector similarity search, and concurrent reads/writes. The Streamlit app would query the DB directly.

**Structured output enforcement**  
Use Anthropic's tool-use / function-calling mode to enforce the JSON schema at the API level — eliminating the need to strip markdown fences or handle malformed output.

**Incremental parsing with change detection**  
Hash each resume file. On re-upload, only re-parse if the hash changed. This avoids redundant API calls.

**Cost estimate at scale**  
Claude Haiku processes ~1,000 tokens per resume at ~$0.00025/1K input tokens. 10,000 resumes ≈ $2.50 to parse the entire corpus — negligible at any reasonable scale.

**Monitoring**  
Log parse success rate, extraction quality (field fill rate), and API latency. A fill rate drop signals either a new resume format or model degradation.

---

*Total implementation time: ~7 hours*